# Lab 8 - CNN trên MNIST và CIFAR-10

In [ ]:
import os
import random
import time
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms, models
from torchvision.transforms import Compose, Resize, ToTensor, Normalize
from PIL import Image

SEED = 42

def set_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = False
    torch.backends.cudnn.benchmark = True

set_seed(SEED)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Dang dung: {device}')

DATA_DIR = Path('/kaggle/working/data') if Path('/kaggle/working').exists() else Path('./data')
DATA_DIR.mkdir(parents=True, exist_ok=True)

## 1. MNIST - tải dữ liệu và quan sát

In [ ]:
mnist_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))
])

train_set = datasets.MNIST(root=str(DATA_DIR), train=True, download=True, transform=mnist_transform)
test_set = datasets.MNIST(root=str(DATA_DIR), train=False, download=True, transform=mnist_transform)

train_loader = DataLoader(train_set, batch_size=64, shuffle=True, num_workers=2, pin_memory=True)
test_loader = DataLoader(test_set, batch_size=64, shuffle=False, num_workers=2, pin_memory=True)

images, labels = next(iter(train_loader))
print(f'Batch images shape: {images.shape}')
print(f'Batch labels shape: {labels.shape}')
print(f'Train size: {len(train_set)}')
print(f'Test size: {len(test_set)}')
print(f'Classes: {train_set.classes}')

fig, axs = plt.subplots(2, 8, figsize=(12, 3.2))
for i, ax in enumerate(axs.flat):
    ax.imshow(images[i, 0], cmap='gray')
    ax.set_title(f'label = {labels[i].item()}')
    ax.axis('off')
plt.tight_layout()
plt.show()

## 2. Định nghĩa mô hình CNN

In [ ]:
class TinyCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(1, 16, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(16, 32, kernel_size=3, padding=1)
        self.pool = nn.MaxPool2d(2, 2)
        self.fc = nn.Linear(32 * 7 * 7, 10)

    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))
        x = self.pool(F.relu(self.conv2(x)))
        x = x.view(x.size(0), -1)
        return self.fc(x)

model = TinyCNN().to(device)
print(model)
print(f'Tong tham so: {sum(p.numel() for p in model.parameters()):,}')

## 3. Hàm train, evaluate và learning curve

In [ ]:
def evaluate(model, loader):
    model.eval()
    correct, total, loss_sum = 0, 0, 0.0
    criterion = nn.CrossEntropyLoss()
    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            logits = model(x)
            loss = criterion(logits, y)
            loss_sum += loss.item() * y.size(0)
            correct += (logits.argmax(1) == y).sum().item()
            total += y.size(0)
    return loss_sum / total, 100 * correct / total


def train_classifier(model, train_loader, test_loader, epochs=3, lr=0.001):
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=lr)
    history = {'train_loss': [], 'train_acc': [], 'test_loss': [], 'test_acc': []}

    init_loss, init_acc = evaluate(model, test_loader)
    print(f'Before training: test_loss = {init_loss:.4f} | test_acc = {init_acc:.2f}%')

    for epoch in range(epochs):
        model.train()
        correct, total, loss_sum = 0, 0, 0.0
        t0 = time.time()
        for x, y in train_loader:
            x, y = x.to(device), y.to(device)
            optimizer.zero_grad()
            logits = model(x)
            loss = criterion(logits, y)
            loss.backward()
            optimizer.step()
            loss_sum += loss.item() * y.size(0)
            correct += (logits.argmax(1) == y).sum().item()
            total += y.size(0)

        train_loss = loss_sum / total
        train_acc = 100 * correct / total
        test_loss, test_acc = evaluate(model, test_loader)
        history['train_loss'].append(train_loss)
        history['train_acc'].append(train_acc)
        history['test_loss'].append(test_loss)
        history['test_acc'].append(test_acc)
        print(f'Epoch {epoch+1:02d}: train_acc = {train_acc:.2f}% | test_acc = {test_acc:.2f}% | time = {time.time()-t0:.1f}s')
    return history


def plot_accuracy(history, title):
    plt.figure(figsize=(7, 4))
    plt.plot(history['train_acc'], 'o-', label='Train accuracy')
    plt.plot(history['test_acc'], 's-', label='Test accuracy')
    plt.xlabel('Epoch')
    plt.ylabel('Accuracy (%)')
    plt.title(title)
    plt.legend()
    plt.grid(alpha=0.3)
    plt.show()

mnist_history = train_classifier(model, train_loader, test_loader, epochs=5, lr=0.001)
plot_accuracy(mnist_history, 'TinyCNN tren MNIST')

## 4. Visualize prediction và ảnh dự đoán sai

In [ ]:
def denorm_mnist(x):
    return x * 0.3081 + 0.1307

model.eval()
images, labels = next(iter(test_loader))
images, labels = images.to(device), labels.to(device)
with torch.no_grad():
    outputs = model(images)
    probs = F.softmax(outputs, dim=1)
    preds = outputs.argmax(1)

fig, axs = plt.subplots(2, 8, figsize=(14, 4))
for i, ax in enumerate(axs.flat):
    img = denorm_mnist(images[i, 0].detach().cpu()).clamp(0, 1).numpy()
    pred, true = preds[i].item(), labels[i].item()
    conf = probs[i, pred].item() * 100
    color = 'green' if pred == true else 'red'
    ax.imshow(img, cmap='gray')
    ax.set_title(f'pred={pred} ({conf:.0f}%)\ntrue={true}', color=color, fontsize=9)
    ax.axis('off')
plt.tight_layout()
plt.show()

wrong_imgs, wrong_pred, wrong_true = [], [], []
model.eval()
with torch.no_grad():
    for x, y in test_loader:
        x, y = x.to(device), y.to(device)
        preds = model(x).argmax(1)
        mask = preds != y
        for img, p, t in zip(x[mask], preds[mask], y[mask]):
            wrong_imgs.append(img.detach().cpu())
            wrong_pred.append(p.item())
            wrong_true.append(t.item())
        if len(wrong_imgs) >= 16:
            break

fig, axs = plt.subplots(2, 8, figsize=(14, 4))
for i, ax in enumerate(axs.flat):
    if i < len(wrong_imgs):
        img = denorm_mnist(wrong_imgs[i][0]).clamp(0, 1).numpy()
        ax.imshow(img, cmap='gray')
        ax.set_title(f'pred={wrong_pred[i]}\ntrue={wrong_true[i]}', color='red', fontsize=9)
    ax.axis('off')
plt.suptitle('Nhung anh MNIST doan sai', color='red')
plt.tight_layout()
plt.show()

## 5. So sánh mạng nông, vừa, sâu

In [ ]:
class ShallowCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv = nn.Conv2d(1, 16, 3, padding=1)
        self.pool = nn.MaxPool2d(2, 2)
        self.fc = nn.Linear(16 * 14 * 14, 10)

    def forward(self, x):
        x = self.pool(F.relu(self.conv(x)))
        return self.fc(x.view(x.size(0), -1))


class DeepCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(1, 16, 3, padding=1)
        self.conv2 = nn.Conv2d(16, 32, 3, padding=1)
        self.conv3 = nn.Conv2d(32, 64, 3, padding=1)
        self.pool = nn.MaxPool2d(2, 2)
        self.fc = nn.Linear(64 * 3 * 3, 10)

    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))
        x = self.pool(F.relu(self.conv2(x)))
        x = self.pool(F.relu(self.conv3(x)))
        return self.fc(x.view(x.size(0), -1))


def quick_train(model_class, epochs=3):
    set_seed(SEED)
    m = model_class().to(device)
    opt = optim.Adam(m.parameters(), lr=0.001)
    crit = nn.CrossEntropyLoss()
    for ep in range(epochs):
        m.train()
        for x, y in train_loader:
            x, y = x.to(device), y.to(device)
            opt.zero_grad()
            loss = crit(m(x), y)
            loss.backward()
            opt.step()
    test_loss, test_acc = evaluate(m, test_loader)
    n_params = sum(p.numel() for p in m.parameters())
    return {'model': model_class.__name__, 'params': n_params, 'test_loss': test_loss, 'test_acc': test_acc}

arch_results = [quick_train(cls, epochs=3) for cls in [ShallowCNN, TinyCNN, DeepCNN]]
for row in arch_results:
    print(f"{row['model']:11s}: params = {row['params']:>8,} | test_acc = {row['test_acc']:.2f}%")

## 6. Augmentation trên MNIST

In [ ]:
transform_aug = transforms.Compose([
    transforms.RandomRotation(degrees=10),
    transforms.RandomAffine(degrees=0, translate=(0.1, 0.1)),
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))
])

raw_aug_set = datasets.MNIST(root=str(DATA_DIR), train=True, download=True, transform=transform_aug)
fig, axs = plt.subplots(1, 8, figsize=(14, 2))
for i, ax in enumerate(axs):
    img, lbl = raw_aug_set[0]
    ax.imshow(denorm_mnist(img[0]).clamp(0, 1), cmap='gray')
    ax.set_title(f'aug #{i+1}', fontsize=8)
    ax.axis('off')
plt.suptitle(f'1 anh so {lbl} qua 8 lan augmentation')
plt.tight_layout()
plt.show()

train_set_aug = datasets.MNIST(root=str(DATA_DIR), train=True, download=True, transform=transform_aug)
train_loader_aug = DataLoader(train_set_aug, batch_size=64, shuffle=True, num_workers=2, pin_memory=True)

set_seed(SEED)
m_aug = TinyCNN().to(device)
mnist_aug_history = train_classifier(m_aug, train_loader_aug, test_loader, epochs=5, lr=0.001)
print(f'TinyCNN + augmentation: test_acc = {mnist_aug_history["test_acc"][-1]:.2f}%')
print(f'TinyCNN khong augmentation: test_acc = {mnist_history["test_acc"][-1]:.2f}%')
plot_accuracy(mnist_aug_history, 'TinyCNN tren MNIST voi augmentation')

## 7. Kernel cổ điển

In [ ]:
def get_sample_image_256():
    cifar_preview = datasets.CIFAR10(root=str(DATA_DIR), train=False, download=True)
    img, label = cifar_preview[3]
    img = img.resize((256, 256)).convert('L')
    return img

img = get_sample_image_256()
img_t = torch.tensor(np.array(img), dtype=torch.float32).unsqueeze(0).unsqueeze(0) / 255.0

kernels = {
    'Identity': torch.tensor([[0, 0, 0], [0, 1, 0], [0, 0, 0]], dtype=torch.float32),
    'Sobel-y': torch.tensor([[-1, -2, -1], [0, 0, 0], [1, 2, 1]], dtype=torch.float32),
    'Sobel-x': torch.tensor([[-1, 0, 1], [-2, 0, 2], [-1, 0, 1]], dtype=torch.float32),
    'Box blur': torch.tensor([[1, 1, 1], [1, 1, 1], [1, 1, 1]], dtype=torch.float32) / 9.0,
    'Sharpen': torch.tensor([[0, -1, 0], [-1, 5, -1], [0, -1, 0]], dtype=torch.float32),
    'Emboss': torch.tensor([[-2, -1, 0], [-1, 1, 1], [0, 1, 2]], dtype=torch.float32),
    'Diagonal': torch.tensor([[0, -1, -2], [1, 0, -1], [2, 1, 0]], dtype=torch.float32),
}

fig, axs = plt.subplots(2, 4, figsize=(14, 7))
axs[0, 0].imshow(img, cmap='gray')
axs[0, 0].set_title('Anh goc')
axs[0, 0].axis('off')

for ax, (name, K) in zip(axs.flat[1:], kernels.items()):
    out_img = F.conv2d(img_t, K.view(1, 1, 3, 3), padding=1).squeeze().numpy()
    ax.imshow(out_img, cmap='gray')
    ax.set_title(name)
    ax.axis('off')

for i in range(len(kernels) + 1, 8):
    axs.flat[i].axis('off')
plt.tight_layout()
plt.show()

## 8. Pooling

In [ ]:
fig, axs = plt.subplots(2, 5, figsize=(14, 6))

xm = img_t.clone()
axs[0, 0].imshow(xm[0, 0], cmap='gray')
axs[0, 0].set_title(f'Goc {xm.shape[-1]}x{xm.shape[-1]}')
for i in range(4):
    xm = F.max_pool2d(xm, 2)
    axs[0, i + 1].imshow(xm[0, 0], cmap='gray')
    axs[0, i + 1].set_title(f'MaxPool x{i+1}\n{xm.shape[-1]}x{xm.shape[-1]}')

xa = img_t.clone()
axs[1, 0].imshow(xa[0, 0], cmap='gray')
axs[1, 0].set_title(f'Goc {xa.shape[-1]}x{xa.shape[-1]}')
for i in range(4):
    xa = F.avg_pool2d(xa, 2)
    axs[1, i + 1].imshow(xa[0, 0], cmap='gray')
    axs[1, i + 1].set_title(f'AvgPool x{i+1}\n{xa.shape[-1]}x{xa.shape[-1]}')

for ax in axs.flat:
    ax.axis('off')
axs[0, 0].set_ylabel('MaxPool', fontsize=12)
axs[1, 0].set_ylabel('AvgPool', fontsize=12)
plt.suptitle('Pooling')
plt.tight_layout()
plt.show()

## 9. Feature map với VGG-16

In [ ]:
def load_vgg16_model():
    try:
        weights = models.VGG16_Weights.IMAGENET1K_V1
        net = models.vgg16(weights=weights).to(device).eval()
        return net, True
    except Exception as e:
        print('Khong tai duoc pretrained weights, dung VGG16 weights=None')
        print(type(e).__name__, e)
        net = models.vgg16(weights=None).to(device).eval()
        return net, False

vgg, pretrained_ok = load_vgg16_model()
print('Pretrained:', pretrained_ok)
print(vgg.features)

sample_rgb = datasets.CIFAR10(root=str(DATA_DIR), train=False, download=True)[3][0].convert('RGB')
transform_imnet = Compose([
    Resize((224, 224)),
    ToTensor(),
    Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])
x = transform_imnet(sample_rgb).unsqueeze(0).to(device)
print('Input shape:', x.shape)

features = {}

def make_hook(name):
    def hook(module, inp, out):
        features[name] = out.detach()
    return hook

handles = [
    vgg.features[0].register_forward_hook(make_hook('block1_conv1')),
    vgg.features[10].register_forward_hook(make_hook('block3_conv1')),
    vgg.features[24].register_forward_hook(make_hook('block5_conv1')),
]

with torch.no_grad():
    _ = vgg(x)

for h in handles:
    h.remove()

for k, v in features.items():
    print(f'{k}: feature map shape = {tuple(v.shape)}')

plt.figure(figsize=(3, 3))
plt.imshow(sample_rgb.resize((224, 224)))
plt.title('Anh dau vao VGG')
plt.axis('off')
plt.show()

In [ ]:
def show_feature_maps(fmap, title, n_show=8):
    fmap = fmap[0].detach().cpu()
    channel_scores = fmap.abs().mean(dim=(1, 2))
    top_idx = channel_scores.argsort(descending=True)[:n_show]
    fig, axs = plt.subplots(1, n_show, figsize=(16, 2.4))
    for i, ax in enumerate(axs):
        ch = top_idx[i].item()
        ax.imshow(fmap[ch], cmap='viridis')
        ax.set_title(f'ch {ch}', fontsize=9)
        ax.axis('off')
    plt.suptitle(title, fontsize=12)
    plt.tight_layout()
    plt.show()

show_feature_maps(features['block1_conv1'], 'block1_conv1')
show_feature_maps(features['block3_conv1'], 'block3_conv1')
show_feature_maps(features['block5_conv1'], 'block5_conv1')

## 10. Filter learned của TinyCNN

In [ ]:
weights = model.conv1.weight.data.detach().cpu()
fig, axs = plt.subplots(2, 8, figsize=(12, 3.5))
for i, ax in enumerate(axs.flat):
    w = weights[i, 0]
    vmax = w.abs().max().item()
    ax.imshow(w, cmap='RdBu_r', vmin=-vmax, vmax=vmax)
    ax.set_title(f'filter {i}', fontsize=8)
    ax.axis('off')
plt.suptitle('16 filter 3x3 cua TinyCNN sau khi train MNIST')
plt.tight_layout()
plt.show()

## 11. Bài mở rộng - CIFAR-10

In [ ]:
cifar_classes = ['airplane', 'automobile', 'bird', 'cat', 'deer', 'dog', 'frog', 'horse', 'ship', 'truck']

cifar_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2470, 0.2435, 0.2616))
])

cifar_train = datasets.CIFAR10(root=str(DATA_DIR), train=True, download=True, transform=cifar_transform)
cifar_test = datasets.CIFAR10(root=str(DATA_DIR), train=False, download=True, transform=cifar_transform)
cifar_train_loader = DataLoader(cifar_train, batch_size=128, shuffle=True, num_workers=2, pin_memory=True)
cifar_test_loader = DataLoader(cifar_test, batch_size=128, shuffle=False, num_workers=2, pin_memory=True)

x_batch, y_batch = next(iter(cifar_train_loader))
print('CIFAR batch shape:', x_batch.shape)
print('Train size:', len(cifar_train))
print('Test size:', len(cifar_test))

mean = torch.tensor((0.4914, 0.4822, 0.4465)).view(3, 1, 1)
std = torch.tensor((0.2470, 0.2435, 0.2616)).view(3, 1, 1)

def denorm_cifar(x):
    return (x.cpu() * std + mean).clamp(0, 1)

fig, axs = plt.subplots(2, 8, figsize=(14, 4))
for i, ax in enumerate(axs.flat):
    img = denorm_cifar(x_batch[i]).permute(1, 2, 0).numpy()
    ax.imshow(img)
    ax.set_title(cifar_classes[y_batch[i].item()], fontsize=8)
    ax.axis('off')
plt.tight_layout()
plt.show()

In [ ]:
class CifarCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.Conv2d(32, 32, 3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),
            nn.Dropout(0.15),
            nn.Conv2d(32, 64, 3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.Conv2d(64, 64, 3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),
            nn.Dropout(0.20),
            nn.Conv2d(64, 128, 3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128 * 4 * 4, 256),
            nn.ReLU(inplace=True),
            nn.Dropout(0.35),
            nn.Linear(256, 10),
        )

    def forward(self, x):
        return self.classifier(self.features(x))

set_seed(SEED)
cifar_model = CifarCNN().to(device)
print(cifar_model)
print(f'Tong tham so: {sum(p.numel() for p in cifar_model.parameters()):,}')

cifar_history = train_classifier(cifar_model, cifar_train_loader, cifar_test_loader, epochs=5, lr=0.001)
plot_accuracy(cifar_history, 'CifarCNN tren CIFAR-10')

In [ ]:
cifar_model.eval()
images, labels = next(iter(cifar_test_loader))
images, labels = images.to(device), labels.to(device)
with torch.no_grad():
    outputs = cifar_model(images)
    probs = F.softmax(outputs, dim=1)
    preds = outputs.argmax(1)

fig, axs = plt.subplots(2, 8, figsize=(15, 4.5))
for i, ax in enumerate(axs.flat):
    img = denorm_cifar(images[i].detach()).permute(1, 2, 0).numpy()
    pred, true = preds[i].item(), labels[i].item()
    conf = probs[i, pred].item() * 100
    color = 'green' if pred == true else 'red'
    ax.imshow(img)
    ax.set_title(f'pred={cifar_classes[pred]} ({conf:.0f}%)\ntrue={cifar_classes[true]}', color=color, fontsize=8)
    ax.axis('off')
plt.tight_layout()
plt.show()

wrong_imgs, wrong_pred, wrong_true = [], [], []
with torch.no_grad():
    for x, y in cifar_test_loader:
        x, y = x.to(device), y.to(device)
        preds = cifar_model(x).argmax(1)
        mask = preds != y
        for img, p, t in zip(x[mask], preds[mask], y[mask]):
            wrong_imgs.append(img.detach().cpu())
            wrong_pred.append(p.item())
            wrong_true.append(t.item())
        if len(wrong_imgs) >= 16:
            break

fig, axs = plt.subplots(2, 8, figsize=(15, 4.5))
for i, ax in enumerate(axs.flat):
    if i < len(wrong_imgs):
        img = denorm_cifar(wrong_imgs[i]).permute(1, 2, 0).numpy()
        ax.imshow(img)
        ax.set_title(f'pred={cifar_classes[wrong_pred[i]]}\ntrue={cifar_classes[wrong_true[i]]}', color='red', fontsize=8)
    ax.axis('off')
plt.suptitle('Nhung anh CIFAR-10 doan sai', color='red')
plt.tight_layout()
plt.show()

## 12. CIFAR-10 với augmentation

In [ ]:
cifar_transform_aug = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2470, 0.2435, 0.2616))
])

cifar_train_aug = datasets.CIFAR10(root=str(DATA_DIR), train=True, download=True, transform=cifar_transform_aug)
cifar_train_loader_aug = DataLoader(cifar_train_aug, batch_size=128, shuffle=True, num_workers=2, pin_memory=True)

preview_aug = datasets.CIFAR10(root=str(DATA_DIR), train=True, download=True, transform=cifar_transform_aug)
fig, axs = plt.subplots(1, 8, figsize=(14, 2))
for i, ax in enumerate(axs):
    img, lbl = preview_aug[0]
    ax.imshow(denorm_cifar(img).permute(1, 2, 0).numpy())
    ax.set_title(f'aug #{i+1}', fontsize=8)
    ax.axis('off')
plt.suptitle('1 anh CIFAR-10 qua 8 lan augmentation')
plt.tight_layout()
plt.show()

set_seed(SEED)
cifar_model_aug = CifarCNN().to(device)
cifar_aug_history = train_classifier(cifar_model_aug, cifar_train_loader_aug, cifar_test_loader, epochs=5, lr=0.001)
plot_accuracy(cifar_aug_history, 'CifarCNN tren CIFAR-10 voi augmentation')

print(f'CIFAR-10 khong augmentation: test_acc = {cifar_history["test_acc"][-1]:.2f}%')
print(f'CIFAR-10 co augmentation: test_acc = {cifar_aug_history["test_acc"][-1]:.2f}%')

## 13. Bảng kết quả

In [ ]:
import pandas as pd

summary_rows = [
    {'experiment': 'MNIST TinyCNN', 'epochs': 5, 'test_acc': mnist_history['test_acc'][-1]},
    {'experiment': 'MNIST TinyCNN + augmentation', 'epochs': 5, 'test_acc': mnist_aug_history['test_acc'][-1]},
    {'experiment': 'CIFAR-10 CifarCNN', 'epochs': 5, 'test_acc': cifar_history['test_acc'][-1]},
    {'experiment': 'CIFAR-10 CifarCNN + augmentation', 'epochs': 5, 'test_acc': cifar_aug_history['test_acc'][-1]},
]
for row in arch_results:
    summary_rows.append({'experiment': f'MNIST {row["model"]}', 'epochs': 3, 'test_acc': row['test_acc'], 'params': row['params']})

summary_df = pd.DataFrame(summary_rows)
display(summary_df)